In [2]:
import rasterio
import numpy as np
from rasterio.warp import reproject, calculate_default_transform, Resampling

# ---- USER INPUTS ----
in_tif   = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
out_tif  = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat_25m.tif"
OUT_RES  = 25.0   # meters per pixel
# If your source is already in meters, keep dst_crs=None.
# If it is in degrees (EPSG:4326), set your projected CRS (e.g., UTM 14N):
dst_crs  = None    # e.g., "EPSG:32614" for Texas UTM 14N
# Aggregation for continuous depth: average (mean). Alternatives: Resampling.max, Resampling.min, Resampling.sum, Resampling.mode
AGG      = Resampling.average
# ---------------------

with rasterio.open(in_tif) as src:
    src_crs = src.crs
    if dst_crs is None:
        dst_crs = src_crs

    # Build target grid at the requested resolution
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=OUT_RES
    )

    meta = src.meta.copy()
    meta.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "compress": "lzw"
    })

    # Propagate nodata if present
    src_nodata = src.nodata
    if src_nodata is not None:
        meta["nodata"] = src_nodata

    print(f"Input res:  {src.res} (units of source CRS)")
    print(f"Output res: {(abs(transform.a), abs(transform.e))} (units of dst CRS)")

    with rasterio.open(out_tif, "w", **meta) as dst:
        for b in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, b),
                destination=rasterio.band(dst, b),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=AGG,
                src_nodata=src_nodata,
                dst_nodata=src_nodata
            )

print("Done:", out_tif)


Input res:  (200.0, 200.0) (units of source CRS)
Output res: (25.0, 25.0) (units of dst CRS)
Done: D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat_25m.tif
